# Optiver Kaggle Submission Notebook

This notebook loads your trained fold checkpoints, runs inference on `test.csv`, and creates a Kaggle-ready `submission.csv`.

In [1]:
# Optional: install dependencies in Colab
%pip install -q timm kaggle

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'
MODEL_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/'
LOCAL_DATA_DIR = '/content/data'

if not os.path.exists(LOCAL_DATA_DIR):
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.exists(f'{LOCAL_DATA_DIR}/test.csv'):
    !cp -r {DATA_DIR} {LOCAL_DATA_DIR}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
class ToImage(object):
    def __init__(self, output_size=(600, 80, 4), bounds='auto', pad=True, include_features=True):
        self.output_size = output_size
        self.bounds = bounds
        self.pad = pad
        self.include_features = include_features

    @staticmethod
    def _safe_std(values: np.ndarray) -> float:
        if values.size < 2:
            return 0.0
        return float(np.std(values, ddof=1))

    def _extract_engineered_features(self, book: pd.DataFrame, trade: pd.DataFrame) -> np.ndarray:
        if book.empty:
            return np.zeros(16, dtype=np.float32)

        bid_price1 = book['bid_price1'].to_numpy(dtype=np.float64)
        ask_price1 = book['ask_price1'].to_numpy(dtype=np.float64)
        bid_size1 = book['bid_size1'].to_numpy(dtype=np.float64)
        ask_size1 = book['ask_size1'].to_numpy(dtype=np.float64)
        bid_size2 = book['bid_size2'].to_numpy(dtype=np.float64)
        ask_size2 = book['ask_size2'].to_numpy(dtype=np.float64)
        seconds = book['seconds_in_bucket'].to_numpy(dtype=np.int32)

        denom = bid_size1 + ask_size1 + 1e-8
        wap1 = (bid_price1 * ask_size1 + ask_price1 * bid_size1) / denom
        spread1 = ask_price1 - bid_price1

        wap1_clipped = np.clip(wap1, 1e-8, None)
        if wap1_clipped.size > 1:
            log_ret = np.diff(np.log(wap1_clipped))
            realized_vol = float(np.sqrt(np.sum(log_ret * log_ret)))
        else:
            realized_vol = 0.0

        vol_l1 = bid_size1 + ask_size1
        vol_l2 = bid_size2 + ask_size2
        vol_total = vol_l1 + vol_l2
        vol_imbalance = (bid_size1 - ask_size1) / (bid_size1 + ask_size1 + 1e-8)

        book_update_count = float(len(book))
        unique_seconds_count = float(np.unique(seconds).size)
        mean_spread = float(np.mean(spread1))
        std_spread = self._safe_std(spread1)
        wap1_mean = float(np.mean(wap1))
        vol_sum = float(np.sum(vol_total))
        vol_q90 = float(np.quantile(vol_total, 0.9)) if vol_total.size else 0.0
        vol_imbalance_mean = float(np.mean(vol_imbalance))
        vol_imbalance_std = self._safe_std(vol_imbalance)

        trade_count = 0.0
        trade_size_sum = 0.0
        trade_order_count_sum = 0.0
        trade_avg_size_per_order = 0.0
        trade_inter_arrival_mean = 0.0
        trade_price_return_std = 0.0

        if not trade.empty:
            trade_count = float(len(trade))
            trade_seconds = trade['seconds_in_bucket'].to_numpy(dtype=np.int32)
            trade_size = trade['size'].to_numpy(dtype=np.float64)
            trade_order_count = trade['order_count'].to_numpy(dtype=np.float64)
            trade_price = trade['price'].to_numpy(dtype=np.float64)

            trade_size_sum = float(np.sum(trade_size))
            trade_order_count_sum = float(np.sum(trade_order_count))
            trade_avg_size_per_order = float(np.mean(trade_size / (trade_order_count + 1e-8)))

            if trade_seconds.size > 1:
                trade_inter_arrival_mean = float(np.mean(np.diff(trade_seconds)))

            if trade_price.size > 1:
                trade_price_ret = np.diff(np.log(np.clip(trade_price, 1e-8, None)))
                trade_price_return_std = self._safe_std(trade_price_ret)

        features = np.array([
            book_update_count, unique_seconds_count, mean_spread, std_spread, wap1_mean, realized_vol,
            vol_sum, vol_q90, vol_imbalance_mean, vol_imbalance_std, trade_count, trade_size_sum,
            trade_order_count_sum, trade_avg_size_per_order, trade_inter_arrival_mean, trade_price_return_std,
        ], dtype=np.float32)
        return features

    def __call__(self, sample):
        book, trade = sample['book'], sample['trade']
        n_time, n_price, n_channels = self.output_size
        image = np.zeros((n_time, n_price, n_channels), dtype=np.int32)
        features = self._extract_engineered_features(book, trade) if self.include_features else None

        if book.empty:
            output = {'image': image}
            if features is not None:
                output['features'] = features
            return output

        if self.bounds == 'auto':
            mid_price = 0.5 * (
                book['bid_price1'].to_numpy(dtype=np.float64) +
                book['ask_price1'].to_numpy(dtype=np.float64)
            )
            price_reference = float(np.median(mid_price))
            if not np.isfinite(price_reference) or price_reference <= 0:
                price_reference = float(np.median(
                    book[['bid_price1', 'bid_price2', 'ask_price1', 'ask_price2']].to_numpy(dtype=np.float64)
                ))
            if not np.isfinite(price_reference) or price_reference <= 0:
                price_reference = 1.0
            price_edges = np.linspace(-0.02, 0.02, n_price + 1)
        else:
            price_reference = 1.0
            price_edges = np.linspace(float(self.bounds[0]), float(self.bounds[1]), n_price + 1)

        bid_bin_1 = np.searchsorted(price_edges, (book['bid_price1'].to_numpy(dtype=np.float64) - price_reference) / price_reference) - 1
        bid_bin_2 = np.searchsorted(price_edges, (book['bid_price2'].to_numpy(dtype=np.float64) - price_reference) / price_reference) - 1
        ask_bin_1 = np.searchsorted(price_edges, (book['ask_price1'].to_numpy(dtype=np.float64) - price_reference) / price_reference) - 1
        ask_bin_2 = np.searchsorted(price_edges, (book['ask_price2'].to_numpy(dtype=np.float64) - price_reference) / price_reference) - 1

        bid_bin_1 = np.clip(bid_bin_1, 0, n_price - 1)
        bid_bin_2 = np.clip(bid_bin_2, 0, n_price - 1)
        ask_bin_1 = np.clip(ask_bin_1, 0, n_price - 1)
        ask_bin_2 = np.clip(ask_bin_2, 0, n_price - 1)

        sec = book['seconds_in_bucket'].to_numpy(dtype=np.int32)
        bs1 = book['bid_size1'].to_numpy(dtype=np.int32)
        bs2 = book['bid_size2'].to_numpy(dtype=np.int32)
        as1 = book['ask_size1'].to_numpy(dtype=np.int32)
        as2 = book['ask_size2'].to_numpy(dtype=np.int32)

        np.add.at(image, (sec, bid_bin_1, 0), bs1)
        np.add.at(image, (sec, bid_bin_2, 0), bs2)
        np.add.at(image, (sec, ask_bin_1, 1), as1)
        np.add.at(image, (sec, ask_bin_2, 1), as2)

        if not trade.empty:
            trade_sec = trade['seconds_in_bucket'].to_numpy(dtype=np.int32)
            trade_bin = np.searchsorted(price_edges, (trade['price'].to_numpy(dtype=np.float64) - price_reference) / price_reference, side='right') - 1
            trade_bin = np.clip(trade_bin, 0, n_price - 1)
            trade_size = trade['size'].to_numpy(dtype=np.int32)
            trade_oc = trade['order_count'].to_numpy(dtype=np.int32)

            np.add.at(image[:, :, 2], (trade_sec, trade_bin), trade_size)
            if self.pad:
                left_mask = trade_bin - 1 >= 0
                right_mask = trade_bin + 1 < n_price
                np.add.at(image[:, :, 2], (trade_sec[left_mask], trade_bin[left_mask] - 1), trade_size[left_mask])
                np.add.at(image[:, :, 2], (trade_sec[right_mask], trade_bin[right_mask] + 1), trade_size[right_mask])
            np.add.at(image[:, :, 3], (trade_sec, trade_bin), trade_oc)

        output = {'image': image}
        if features is not None:
            output['features'] = features
        return output

In [4]:
class OrderFlowRegressor(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()
        self.image_encoder = timm.create_model(
            'vit_base_patch16_224',
            pretrained=False,
            in_chans=4,
            num_classes=0,
            global_pool='avg',
            dynamic_img_size=True
        )
        image_dim = self.image_encoder.num_features

        self.feature_encoder = nn.Sequential(
            nn.LayerNorm(tabular_dim),
            nn.Linear(tabular_dim, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128),
            nn.GELU(),
        )

        self.regression_head = nn.Sequential(
            nn.Linear(image_dim + 128, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, image, features):
        h, w = image.shape[2], image.shape[3]
        pad_h = (16 - h % 16) % 16
        pad_w = (16 - w % 16) % 16
        image = F.pad(image, (0, pad_w, 0, pad_h))

        image_embedding = self.image_encoder(image)
        feature_embedding = self.feature_encoder(features)
        fused = torch.cat([image_embedding, feature_embedding], dim=1)
        return F.softplus(self.regression_head(fused).squeeze(-1))

In [5]:
class TestOrderFlowDataset(Dataset):
    def __init__(self, test_csv_path, book_path, trade_path, transform=None):
        self.test_df = pd.read_csv(test_csv_path)
        self.index_map = self.test_df[['stock_id', 'time_id']].to_numpy()
        self.row_ids = self.test_df['row_id'].astype(str).to_numpy()

        self.transform = transform

        full_book = pd.read_parquet(book_path)
        full_trade = pd.read_parquet(trade_path)

        self.books = {k: v for k, v in full_book.groupby(['stock_id', 'time_id'])}
        self.trades = {k: v for k, v in full_trade.groupby(['stock_id', 'time_id'])}

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        stock_id, time_id = self.index_map[idx]
        row_id = self.row_ids[idx]

        sample = {
            'book': self.books.get((stock_id, time_id), pd.DataFrame()),
            'trade': self.trades.get((stock_id, time_id), pd.DataFrame()),
        }

        if self.transform:
            sample = self.transform(sample)

        sample['row_id'] = row_id
        return sample

In [8]:
def infer_with_checkpoint(model_path, test_dataset, batch_size=256, num_workers=2):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Checkpoints are generated in this project and contain numpy arrays; use full unpickling.
    ckpt = torch.load(model_path, map_location=device, weights_only=False)

    feature_dim = int(ckpt['feature_dim'])
    img_mean = ckpt['img_mean'].astype(np.float32)
    img_std = ckpt['img_std'].astype(np.float32)
    feat_mean = ckpt['feat_mean'].astype(np.float32)
    feat_std = ckpt['feat_std'].astype(np.float32)

    model = OrderFlowRegressor(tabular_dim=feature_dim).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    def collate_fn(batch):
        row_ids = [x['row_id'] for x in batch]

        images = []
        features = []
        for x in batch:
            img = np.log1p(x['image'].astype(np.float32))
            img = (img - img_mean[None, None, :]) / img_std[None, None, :]
            img = np.transpose(img, (2, 0, 1)).copy()
            feat = x['features'].astype(np.float32)
            feat = (feat - feat_mean) / feat_std
            images.append(img)
            features.append(feat)

        images = torch.from_numpy(np.stack(images, axis=0))
        features = torch.from_numpy(np.stack(features, axis=0))
        return row_ids, images, features

    loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collate_fn
    )

    all_row_ids = []
    all_preds = []

    with torch.no_grad():
        for row_ids, images, features in loader:
            images = images.to(device, dtype=torch.float32, non_blocking=True)
            features = features.to(device, dtype=torch.float32, non_blocking=True)
            preds = model(images, features).detach().cpu().numpy()
            all_row_ids.extend(row_ids)
            all_preds.append(preds)

    all_preds = np.concatenate(all_preds, axis=0)
    all_preds = np.clip(np.expm1(all_preds), 0.0, None)
    return pd.DataFrame({'row_id': all_row_ids, 'target': all_preds})

In [9]:
# Build test dataset
test_dataset = TestOrderFlowDataset(
    test_csv_path=f'{LOCAL_DATA_DIR}/test.csv',
    book_path=f'{LOCAL_DATA_DIR}/book_test.parquet',
    trade_path=f'{LOCAL_DATA_DIR}/trade_test.parquet',
    transform=ToImage(output_size=(600, 80, 4), include_features=True)
)

# Load all best fold models and ensemble by averaging predictions
checkpoint_paths = sorted(glob.glob(f'{MODEL_DIR}/best_model_fold_*.pth'))
if not checkpoint_paths:
    raise FileNotFoundError('No fold checkpoints found. Train first and verify best_model_fold_*.pth exists.')

print('Found checkpoints:')
for p in checkpoint_paths:
    print(' -', p)

pred_dfs = []
for p in checkpoint_paths:
    pred_df = infer_with_checkpoint(p, test_dataset, batch_size=256, num_workers=2)
    pred_df = pred_df.sort_values('row_id').reset_index(drop=True)
    pred_dfs.append(pred_df)

submission = pred_dfs[0].copy()
stacked = np.stack([d['target'].to_numpy() for d in pred_dfs], axis=0)
submission['target'] = stacked.mean(axis=0)

submission_path = '/content/submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Submission file written to: {submission_path}')
submission.head()

/tmp/ipykernel_47411/4172732950.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.books = {k: v for k, v in full_book.groupby(['stock_id', 'time_id'])}
/tmp/ipykernel_47411/4172732950.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.trades = {k: v for k, v in full_trade.groupby(['stock_id', 'time_id'])}


Found checkpoints:
 - /content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model_fold_0.pth
 - /content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model_fold_1.pth
 - /content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model_fold_2.pth
 - /content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model_fold_3.pth
 - /content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/best_model_fold_4.pth
Submission file written to: /content/submission.csv


,row_id,target
0,0-32,-0.003830
1,0-34,-0.003830
2,0-4,0.002449


In [ ]:
# Download to local machine
from google.colab import files
files.download('/content/submission.csv')

In [10]:
# Submit directly from Colab using Kaggle API.
import os
import glob
import shutil
import subprocess
from datetime import datetime

competition = 'optiver-realized-volatility-prediction'
submission_file = '/content/submission.csv'

if not os.path.exists(submission_file):
    raise FileNotFoundError(f'{submission_file} not found. Run inference cell first.')

kaggle_candidates = sorted(glob.glob('/content/drive/MyDrive/**/kaggle.json', recursive=True))
if not kaggle_candidates:
    raise FileNotFoundError('Could not find kaggle.json anywhere under /content/drive/MyDrive')

kaggle_json_path = kaggle_candidates[0]
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy2(kaggle_json_path, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

message = f"vit-features-5fold-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
cmd = [
    'kaggle', 'competitions', 'submit',
    '-c', competition,
    '-f', submission_file,
    '-m', message,
]

print(f'Using kaggle.json: {kaggle_json_path}')
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'Kaggle submission failed with exit code {result.returncode}')

print('Submission completed successfully.')

/tmp/ipykernel_47411/376207047.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  message = f"vit-features-5fold-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"


Using kaggle.json: /content/drive/MyDrive/kaggle.json
Running: kaggle competitions submit -c optiver-realized-volatility-prediction -f /content/submission.csv -m vit-features-5fold-20260415-030316
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


  0%|          | 0.00/69.0 [00:00<?, ?B/s]
100%|██████████| 69.0/69.0 [00:00<00:00, 84.0B/s]



RuntimeError: Kaggle submission failed with exit code 1

In [11]:
# Diagnostic: check competition status and recent submissions visibility.
import subprocess

for cmd in [
    ['kaggle', 'competitions', 'list', '-s', 'optiver-realized-volatility-prediction'],
    ['kaggle', 'competitions', 'submissions', '-c', 'optiver-realized-volatility-prediction'],
]:
    print('\n$ ' + ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('exit_code=', result.returncode)


$ kaggle competitions list -s optiver-realized-volatility-prediction
ref                                                                              deadline             category        reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  ---------  -----------  ---------  --------------  
https://www.kaggle.com/competitions/optiver-realized-volatility-prediction       2022-01-10 18:36:00  Featured   100,000 Usd       3852            True  
https://www.kaggle.com/competitions/optiver-sjtu-realized-volatility-prediction  2022-05-26 23:59:00  Community        Kudos          0           False  

exit_code= 0

$ kaggle competitions submissions -c optiver-realized-volatility-prediction
fileName        date                        description                                                       status                     publicScore  privateScore  
--------------  --------------------------  --------------